<a href="https://colab.research.google.com/github/santhoshkumar81740/Information-and-Retrieval-Systems-Lab/blob/main/Text_Document_Clustering_using_K_Means.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Text Document Clustering using K-Means

import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix

# ---------------------------------------------------
# 1. Load Standard Dataset
# ---------------------------------------------------

categories = [
    'comp.graphics',
    'rec.sport.baseball',
    'sci.space'
]

data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

documents = data.data
true_labels = data.target
class_names = data.target_names

print("Number of documents:", len(documents))
print("Categories:", class_names)

# ---------------------------------------------------
# 2. Convert Text into TF-IDF Vectors
# ---------------------------------------------------

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", X.shape)

# ---------------------------------------------------
# 3. Apply K-Means Clustering
# ---------------------------------------------------

k = len(categories)

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

# ---------------------------------------------------
# 4. Map Clusters to Actual Classes
# ---------------------------------------------------

# Find the best cluster-to-class mapping
from scipy.optimize import linear_sum_assignment

cm = confusion_matrix(true_labels, cluster_labels)

row_ind, col_ind = linear_sum_assignment(-cm)

mapping = {cluster: true_class
           for true_class, cluster in zip(row_ind, col_ind)}

predicted_labels = np.array([
    mapping[cluster]
    for cluster in cluster_labels
])

# ---------------------------------------------------
# 5. Calculate Purity
# ---------------------------------------------------

purity = np.sum(
    np.max(cm, axis=0)
) / np.sum(cm)

# ---------------------------------------------------
# 6. Calculate Precision, Recall and F-measure
# ---------------------------------------------------

precision_values = []
recall_values = []
f_values = []

for cls in range(k):

    TP = np.sum(
        (predicted_labels == cls) &
        (true_labels == cls)
    )

    FP = np.sum(
        (predicted_labels == cls) &
        (true_labels != cls)
    )

    FN = np.sum(
        (predicted_labels != cls) &
        (true_labels == cls)
    )

    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0

    f_measure = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) else 0
    )

    precision_values.append(precision)
    recall_values.append(recall)
    f_values.append(f_measure)

# ---------------------------------------------------
# 7. Display Results
# ---------------------------------------------------

print("\nPerformance Measures")
print("--------------------")

print("Purity     :", round(purity, 4))
print("Precision  :", round(np.mean(precision_values), 4))
print("Recall     :", round(np.mean(recall_values), 4))
print("F-measure  :", round(np.mean(f_values), 4))

print("\nClass-wise Results")

for i, name in enumerate(class_names):
    print(
        name,
        "Precision =", round(precision_values[i], 4),
        "Recall =", round(recall_values[i], 4),
        "F-measure =", round(f_values[i], 4)
    )

Number of documents: 1774
Categories: ['comp.graphics', 'rec.sport.baseball', 'sci.space']
TF-IDF matrix shape: (1774, 5000)

Performance Measures
--------------------
Purity     : 0.6984
Precision  : 0.8107
Recall     : 0.6989
F-measure  : 0.6971

Class-wise Results
comp.graphics Precision = 0.907 Recall = 0.6849 F-measure = 0.7805
rec.sport.baseball Precision = 0.9891 Recall = 0.4556 F-measure = 0.6239
sci.space Precision = 0.5359 Recall = 0.9562 F-measure = 0.6869
